# CAMPUS-FIND — DBMS Demonstration Walkthrough
### Smart Lost & Found Management System (Hybrid MySQL + MongoDB Architecture)

This beginner-friendly Jupyter notebook demonstrates how **CAMPUS-FIND** uses both **MySQL** (Relational DBMS) and **MongoDB** (Document DBMS) together in a cohesive hybrid architecture.

### Key Demonstrations in this Notebook:
1. **Independent Database Connections** to MySQL and MongoDB.
2. **MongoDB Nested Field Query** on rich item attributes and campus zones.
3. **MySQL Relational Multi-Table JOIN** (claims + users + handoffs).
4. **MySQL Multi-Row ACID Transaction** with commit and rollback verification.
5. **Integrated Hybrid Workflow** linking MongoDB reports to MySQL claims using shared `item_uuid`.

In [ ]:
# 0. Setup and Imports
import sys, os
sys.path.append(os.path.abspath('.'))

import db_mysql
import db_mongo
import services
import seed_data

print('Modules loaded successfully!')

In [ ]:
# Re-seed sample campus data to ensure clean demonstration state
seed_data.seed_all()

---
## 1. Independent Database Connections
Verify that MySQL and MongoDB connect independently from Python.

In [ ]:
mysql_ok, mysql_msg = db_mysql.test_mysql_connection()
mongo_ok, mongo_msg = db_mongo.test_mongo_connection()

print(f"MySQL Status:    [{'OK' if mysql_ok else 'FAIL'}] -> {mysql_msg}")
print(f"MongoDB Status:  [{'OK' if mongo_ok else 'FAIL'}] -> {mongo_msg}")

---
## 2. MongoDB Nested Field Queries
MongoDB excels at storing semi-structured data with flexible nested subdocuments.
Here we query against nested fields: `location.campus_zone`, `nested_attributes.color`, and `urgency_info.is_urgent`.

In [ ]:
# Nested Query A: Find all reports in 'Central Library'
library_items = db_mongo.query_by_nested_location(campus_zone='Central Library')
print(f"Found {len(library_items)} reports in 'Central Library':")
for it in library_items:
    print(f"  - [{it['report_type'].upper()}] {it['title']} (Zone: {it['location']['campus_zone']}, Building: {it['location']['building']})")

print("\nNested Query B: Find all items with color 'Gray':")
gray_items = db_mongo.query_by_nested_attribute(color='Gray')
for it in gray_items:
    print(f"  - {it['title']} | Color: {it['nested_attributes'].get('color')} | Brand: {it['nested_attributes'].get('brand')}")

print("\nNested Query C: Find all items marked URGENT:")
urgent_items = db_mongo.query_urgent_items()
for it in urgent_items:
    print(f"  - ⚡ {it['title']} -> Reason: {it['urgency_info'].get('urgency_reason')}")

---
## 3. MySQL Relational JOIN Demonstration
MySQL handles structured relations, integrity constraints, and multi-table queries.
This query joins `claims` with `users` (as claimant), `users` (as reviewer), and `handoffs` (for safe return token).

In [ ]:
joined_claims = db_mysql.get_claims_with_joins()
print(f"Total Claims with JOIN: {len(joined_claims)}\n")
for c in joined_claims:
    print(f"Claim ID: #{c['claim_id']} | Status: {c['claim_status']}")
    print(f"  Item Snapshot:       {c['snapshot_item_title']} ({c['snapshot_location_zone']})")
    print(f"  Claimant (User JOIN):{c['claimant_name']} <{c['claimant_email']}> ({c['claimant_department']})")
    print(f"  Reviewer JOIN:       {c['reviewer_name'] or 'Awaiting Review'}")
    print(f"  Handoff Token:       {c.get('handoff_code') or 'Pending Approval'}")
    print('-' * 60)

---
## 4. MySQL Multi-Row ACID Transaction (Commit & Rollback)
When a campus moderator approves an ownership claim, multiple tables must update together atomically:
1. Update `claims` table to status = `approved` and record reviewer notes.
2. Insert a new record in `handoffs` with a unique 6-character return code.
3. Insert a student notification in `notifications`.

If any statement fails, all changes are rolled back to maintain consistency.

In [ ]:
# Step 4A: Create a fresh pending claim
import uuid
new_claim_uuid = f"clm-demo-{uuid.uuid4().hex[:6]}"
new_claim_id = db_mysql.execute_write(
    """INSERT INTO claims (claim_uuid, item_uuid, claimant_id, claim_status, private_verification_text,
                          snapshot_item_title, snapshot_report_type, snapshot_location_zone)
       VALUES (%s, 'cf-item-003', 2, 'pending_review', 'Has red silicone tag with high school logo',
               'Dorm Room Brass Keys with Lanyard', 'found', 'South Residence')""",
    (new_claim_uuid,)
)
print(f"Created test claim #{new_claim_id} with status 'pending_review'.")

# Step 4B: Execute atomic transaction to approve claim
approval_res = services.approve_claim_transaction(
    claim_id=new_claim_id,
    moderator_id=3,
    reviewer_notes='Approved in Jupyter DBMS walkthrough: Secret detail matches!'
)
print('\n[ACID Transaction Success]')
print(f"  Handoff Token:   {approval_res['handoff_code']}")
print(f"  Pickup Location: {approval_res['pickup_location']}")

# Step 4C: Rollback demonstration on deliberate SQL failure
try:
    db_mysql.execute_transaction([
        ("UPDATE claims SET reviewer_notes = 'Tampered Notes' WHERE id = %s", (new_claim_id,)),
        ("INSERT INTO invalid_table_that_does_not_exist VALUES ('err')", ())
    ])
except Exception as err:
    print(f'\n[ACID Rollback Caught Expected Exception]: {err}')

# Confirm data was not corrupted by the failed transaction
check_claim = db_mysql.execute_query('SELECT reviewer_notes FROM claims WHERE id = %s', (new_claim_id,))[0]
print(f"  Verified notes in DB: '{check_claim['reviewer_notes']}' (Unchanged by failed transaction!)")

---
## 5. Integrated Hybrid Workflow (Shared `item_uuid`)
Demonstrates how an application workflow queries MongoDB for the item document and enriches it with MySQL relational records using the shared `item_uuid`.

In [ ]:
target_uuid = 'cf-item-001'

# 1. Fetch document from MongoDB
mongo_item = db_mongo.find_item_by_uuid(target_uuid)

# 2. Fetch related relational records from MySQL
mysql_claims = db_mysql.execute_query('SELECT * FROM claims WHERE item_uuid = %s', (target_uuid,))
mysql_handoffs = db_mysql.execute_query('SELECT * FROM handoffs WHERE item_uuid = %s', (target_uuid,))
mysql_matches = db_mysql.execute_query(
    'SELECT * FROM match_suggestions WHERE lost_item_uuid = %s OR found_item_uuid = %s',
    (target_uuid, target_uuid)
)

print('=== COMBINED HYBRID RECORD ===')
print(f'Item UUID (Bridge Key): {target_uuid}')
print(f"Title (from MongoDB):   {mongo_item['title']}")
print(f"Zone (from MongoDB):    {mongo_item['location']['campus_zone']}")
print(f"Brand (from MongoDB):   {mongo_item['nested_attributes']['brand']}")
print(f'MySQL Claims Linked:    {len(mysql_claims)} record(s)')
print(f'MySQL Handoffs Linked:  {len(mysql_handoffs)} record(s)')
print(f'MySQL Matches Linked:   {len(mysql_matches)} record(s)')